# 04.07 Image Forgery Screening (ORB / SIFT & Copy-Move Forgery Detection)

**Phase 04 — Document Processing & OCR (Track A: Document Fraud Screening)**
**Approved Components:** OpenCV, ORB / SIFT Visual Feature Matching, Copy-Move Forgery Detection (CMFD)
**Strict Exclusion:** Siamese Network is **excluded** from the active architecture.

### Research Objectives
1. Preprocess authorized or synthetic document images using OpenCV (grayscale conversion, CLAHE contrast enhancement).
2. Extract local visual keypoints and descriptors using **ORB** and/or **SIFT**.
3. Screen for potentially duplicated or cloned regions via **Copy-Move Forgery Detection (CMFD)**.
4. Visualize suspicious matched clusters and record localization metadata for **MSW manual review**.

### Inputs & Expected Outputs
- **Inputs:** Preprocessed document images from `storage/uploads/originals/` or synthetic forgery test sets.
- **Outputs:** Visual match overlays, CMFD cluster counts, and JSON screening reports (never automatic rejection).


In [ ]:
import sys
from pathlib import Path
from typing import Dict, Any, List, Tuple
import cv2
import numpy as np

PROJECT_ROOT = Path("../../../..").resolve()
if str(PROJECT_ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.services.document_screening.forgery_detection.copy_move_detection import CopyMoveDetector
from app.services.document_screening.forgery_detection.feature_matching import VisualFeatureMatcher
print(f"Project Root: {PROJECT_ROOT}")


In [ ]:
def run_forgery_screening(image_path: Path) -> Dict[str, Any]:
    """Screens a document image for copy-move duplication indicators."""
    if not image_path.exists():
        raise FileNotFoundError(f"Document image not found: {image_path}")
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"Unable to decode image: {image_path}")

    detector = CopyMoveDetector(block_size=16, min_distance=30.0)
    is_suspicious, cluster_count, coords = detector.screen_image(img)
    return {
        "filename": image_path.name,
        "is_suspicious": is_suspicious,
        "cmfd_cluster_count": cluster_count,
        "manual_verification_required": is_suspicious,
        "note": "Visual similarity is an indicator only, not definitive proof of forgery."
    }


### Limitations & Ethical Considerations
- **No Definitive Proof:** ORB/SIFT and CMFD detect visual similarity across regions; repeated clean text or standard form gridlines may trigger false positives.
- **No Automatic Rejection:** Flagged documents must be routed to an authorized MSW for physical/manual verification.
